In [ ]:
!pip install scanpy
!pip install anndata
!pip3 install igraph
!pip install celltypist
!pip install decoupler
!pip install fa2-modified
!pip install louvain


In [ ]:
#Import core single cell datasets

import scanpy as sc
import anndata as ad
import numpy as np


In [ ]:
!wget https://ftp.ncbi.nlm.nih.gov/geo/series/GSE166nnn/GSE166766/suppl/GSE166766%5Fscv2%5F200428.h5ad.gz

In [ ]:
bm_mtx = sc.read_h5ad('/content/497ab773-4fd5-4263-9fdf-7b42a68f1351.h5ad')
bm_mtx.var_names_make_unique()

In [ ]:
bm_mtx = sc.read_h5ad('/content/GSE166766_scv2_200428.h5ad.gz')
bm_mtx.var_names_make_unique()

In [ ]:
#!wget https://datasets.cellxgene.cziscience.com/fdf57c52-ad71-4004-9db2-a962e849b524.h5ad
!wget https://datasets.cellxgene.cziscience.com/497ab773-4fd5-4263-9fdf-7b42a68f1351.h5ad

In [ ]:
"""
mkdir -p GSM6229474
mkdir -p GSM6229475
mv GSM6229474_* GSM6229474
mv GSM6229475_* GSM6229475
cd GSM6229474
mv GSM6229474_C57IMQD_barcodes.tsv.gz barcodes.tsv.gz
mv GSM6229474_C57IMQD_features.tsv.gz features.tsv.gz
mv GSM6229474_C57IMQD_matrix.mtx.gz matrix.mtx.gz
cd ..
cd GSM6229475
mv GSM6229475_C57IMQE_barcodes.tsv.gz barcodes.tsv.gz
mv GSM6229475_C57IMQE_features.tsv.gz features.tsv.gz
mv GSM6229475_C57IMQE_matrix.mtx.gz matrix.mtx.gz
cd ..
"""

In [ ]:
bm_mtx = sc.read_h5ad('/content/497ab773-4fd5-4263-9fdf-7b42a68f1351.h5ad')
bm_mtx.var_names_make_unique()

In [ ]:
bm_mtx

In [ ]:
del bm_mtx.obs['cell_type']

In [ ]:
bm_mtx

In [ ]:
bm_mtx.write_h5ad("bone_marrow.h5ad")

In [ ]:
short_read = sc.read_h5ad('bone_marrow.h5ad')

In [ ]:
short_read

In [ ]:
bm_mtx.shape

In [ ]:
# A useful step for older datasets
bm_mtx.var_names_make_unique()
bm_mtx.obs_names_make_unique()

In [ ]:
bm_mtx.var.head()

In [ ]:
bm_mtx.shape

In [ ]:
bm_mtx.var['MT'] = bm_mtx.var_names.str.startswith("MT-")
bm_mtx.var['RIBO'] = bm_mtx.var_names.str.startswith("RPS", "RPL")
bm_mtx.var['HB'] = bm_mtx.var_names.str.startswith("^HB[^(P)]")

In [ ]:
sc.pp.calculate_qc_metrics(
    bm_mtx, qc_vars=["MT", 'RIBO', 'HB'], inplace=True, log1p=True
)

In [ ]:
import matplotlib.pyplot as plt

plt.rcParams["figure.figsize"] = (5,3)  # Adjust figure size
plt.rcParams["axes.grid"] = True  # Add grid to plots
plt.rcParams["axes.edgecolor"] = "black" # Set plot border color
plt.rcParams["axes.linewidth"] = 1.5 # Set plot border width
plt.rcParams["axes.facecolor"] = "white" # Set background color
plt.rcParams["axes.labelcolor"] = "black" # Set label color
plt.rcParams["xtick.color"] = "black" # Set x-axis tick color
plt.rcParams["ytick.color"] = "black" # Set y-axis tick color
plt.rcParams["text.color"] = "black" # Set text color

In [ ]:
sc.pl.violin(
    bm_mtx,
    ["n_genes_by_counts", 'total_counts', 'pct_counts_MT'],
    jitter=0.4,
    multi_panel=False,
)

In [ ]:
bm_mtx.obs_keys()

In [ ]:
sc.pl.scatter(bm_mtx, "total_counts", "n_genes_by_counts")#, color="cell_type")

In [ ]:
#sc.pp.scrublet(bm_mtx)

In [ ]:
#Normalisation
bm_mtx.layers["counts"] = bm_mtx.X.copy()
sc.pp.normalize_total(bm_mtx)
sc.pp.log1p(bm_mtx)

In [ ]:
#Feature selection
sc.pp.highly_variable_genes(bm_mtx, n_top_genes=1000)
sc.pl.highly_variable_genes(bm_mtx)

In [ ]:
#Dim Reduction
sc.tl.pca(bm_mtx)
sc.pl.pca_variance_ratio(bm_mtx, n_pcs=10, log=False)


In [ ]:
bm_mtx.var_names

In [ ]:
#il_genes = [gene for gene in bm_mtx.var_names[bm_mtx.var['highly_variable']] if gene.lower().startswith('cc')]
#print(il_genes)

In [ ]:
bm_mtx

In [ ]:
sc.pl.pca(bm_mtx,
          #color=["cell_type"],
          cmap="coolwarm")

In [ ]:
sc.pp.neighbors(bm_mtx)
sc.tl.umap(bm_mtx)

In [ ]:
sc.pl.umap(
    bm_mtx,
   #color=["cell_type"],
    size=8,
)

In [ ]:
sc.tl.leiden(bm_mtx, flavor="igraph", n_iterations=10, key_added="leiden_res_", resolution=0.5 )

In [ ]:
sc.pl.umap(
    bm_mtx,
    color=["leiden_res_"],
    size=8,
)

In [ ]:
import decoupler as dc

In [ ]:
!wget wget -O result.txt 'http://www.ensembl.org/biomart/martservice?query=<?xml version="1.0" encoding="UTF-8"?><!DOCTYPE Query><Query  virtualSchemaName = "default" formatter = "CSV" header = "0" uniqueRows = "0" count = "" datasetConfigVersion = "0.6" ><Dataset name = "hsapiens_gene_ensembl" interface = "default" ><Attribute name = "ensembl_gene_id" /><Attribute name = "external_gene_name" /></Dataset></Query>'

In [ ]:
import pandas as pd

ensembl_var = pd.read_csv('/content/result.txt', header = None)

ensembl_var.columns = ['ensembl_gene_id', 'gene_name']

ensembl_var.head(3)

In [ ]:
markers = dc.op.resource(name="PanglaoDB", organism="human")


In [ ]:
markers.head()

In [ ]:
markers.organ.unique()

In [ ]:
# Query Omnipath and get PanglaoDB
markers = dc.op.resource(name="PanglaoDB", organism="human")

# Keep canonical cell type markers alone
#markers = markers[markers["canonical_marker"]]

# Remove duplicated entries
markers = markers[~markers.duplicated(["cell_type", "genesymbol"])]

#Format because dc only accepts cell_type and genesymbol

markers = markers.rename(columns={"cell_type": "source", "genesymbol": "target"})
markers = markers[["source", "target"]]


markers.head()

In [ ]:
#correct target to ensemble
markers = markers.merge(ensembl_var, left_on="target", right_on="gene_name", how="left")
markers = markers.drop(columns=["target"])
# Remove duplicated entries
markers = markers[~markers.duplicated(["source", "ensembl_gene_id"])]

#Format because dc only accepts cell_type and genesymbol
markers = markers.rename(columns={"source": "source", "ensembl_gene_id": "target"})

markers = markers[["source", "target"]]
markers = markers.dropna()

markers.head()

In [ ]:
bm_mtx.var_names

In [ ]:
dc.mt.ulm(data=bm_mtx,
          net=markers,
          tmin = 3)

In [ ]:
score = dc.pp.get_obsm(bm_mtx, key="score_ulm")

In [ ]:
bm_mtx.obsm["score_ulm"].head(1)

In [ ]:
bm_mtx.obsm["score_ulm"].columns

In [ ]:
#rank genes
bm_gene_rank = dc.tl.rankby_group(score, groupby="leiden_res_", reference="rest", method="t-test_overestim_var")
bm_gene_rank = bm_gene_rank[bm_gene_rank["stat"] > 0]
bm_gene_rank.head(5)

In [ ]:
top_names_per_group = bm_gene_rank.groupby('group')['name'].apply(lambda x: x.head(1))
display(top_names_per_group)

In [ ]:
sc.pl.umap(score, color=["Neurons","leiden_res_"], cmap="RdBu_r")

In [ ]:
n_ctypes = 3
ctypes_dict = bm_gene_rank.groupby("group").head(n_ctypes).groupby("group")["name"].apply(lambda x: list(x)).to_dict()
ctypes_dict

In [ ]:
dict_ann = bm_gene_rank[bm_gene_rank["stat"] > 0].groupby("group").head(1).set_index("group")["name"].to_dict()
dict_ann

In [ ]:
dict_ann_unique = {k: v + '_' + str(k) for k, v in dict_ann.items()}
display(dict_ann_unique)

In [ ]:
bm_mtx.obs["leiden_res_"] = bm_mtx.obs["leiden_res_"].cat.rename_categories(dict_ann_unique)

In [ ]:

plt.rcParams["figure.figsize"] = (5,3)  # Adjust figure size
# Reduce the default font size for text elements in plots
plt.rcParams['font.size'] = 8
plt.rcParams['axes.labelsize'] = 8
plt.rcParams['xtick.labelsize'] = 8
plt.rcParams['ytick.labelsize'] = 8

# Now replot the UMAP
sc.pl.umap(
    adata=bm_mtx,
    color=[ "leiden_res_"],#, 'cell_type'],
    ncols=2,
    legend_loc = 'on data',
    size=4,
)

In [ ]:
#Pseudotime inference
sc.tl.diffmap(bm_mtx)

In [ ]:
plt.rcParams["figure.figsize"] = (4,4)
plt.rcParams['axes.labelsize'] = 10

root_ixs = bm_mtx.obsm["X_diffmap"][:, 3].argmin()
sc.pl.scatter(
    bm_mtx,
    basis="diffmap",
    color=["leiden_res_"],
    components=[2, 3],
    size=50
)

bm_mtx.uns["iroot"] = root_ixs

In [ ]:
sc.tl.dpt(bm_mtx)

In [ ]:
sc.pl.scatter(
    bm_mtx,
    basis="umap",
    color=["dpt_pseudotime", 'leiden_res_'],
    color_map="coolwarm",
)

In [ ]:
#Trajectory analysis
root_celltype = "Monocytes_8"


sc.pp.neighbors(bm_mtx, use_rep="X_diffmap")
sc.tl.paga(bm_mtx, groups="leiden_res_")
iroot = np.flatnonzero(bm_mtx.obs["leiden_res_"] == root_celltype)[0]
bm_mtx.uns["iroot"] = iroot
sc.tl.dpt(bm_mtx)

In [ ]:
sc.pl.paga(bm_mtx, color=["leiden_res_"])
sc.pl.tsne(bm_mtx, color=["leiden_res_", "dpt_pseudotime"], legend_loc="on data")

In [ ]:
sc.pl.paga_compare(bm_mtx, threshold=0.001, frameon=True, edges=True)

In [ ]:
sc.pl.draw_graph(bm_mtx, color=['dpt_pseudotime'], legend_loc='on data')
